<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Kaggle download
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/data/ulb --unzip
!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip

Kaggle username: Subbareddy_Subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 143MB/s]

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:03<00:00, 69.4MB/s]



In [2]:

# 2. Clone the repo (already has split.py and utils.py inside it)
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 70 (delta 34), reused 18 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 346.41 KiB | 17.32 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [3]:

# 3. Move in, confirm, move data in
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mv /content/data/ulb /content/data/sparkov data/


/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
import sys
sys.path.append('/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-')
from src.split import split_sparkov, split_ulb

sparkov_train, sparkov_test = split_sparkov()
ulb_train, ulb_test = split_ulb()


Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
Train: 226,980 rows, from 0.0 to 145233.0
Test:  56,746 rows, from 145234.0 to 172792.0


In [5]:

print("Already in namespace:", [v for v in ["sparkov_train", "sparkov_test", "ulb_train", "ulb_test"] if v in dir()])

from src.split import split_sparkov
sparkov_train, sparkov_test = split_sparkov()

print(sparkov_train.shape, sparkov_test.shape)
print(sparkov_train.columns.tolist())
sparkov_train.head(2)

Already in namespace: []
Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1296675, 23) (555719, 23)
['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0


In [6]:
# clean up + naive Sparkov baseline (raw numeric features only)

# 1. Drop the leftover index column - it's not a real feature
sparkov_train = sparkov_train.drop(columns=["Unnamed: 0"])
sparkov_test = sparkov_test.drop(columns=["Unnamed: 0"])

# 2. Sanity check - fraud rate should match the EDA numbers
print("Train fraud rate:", sparkov_train["is_fraud"].mean())
print("Test fraud rate:", sparkov_test["is_fraud"].mean())

# 3. Pick the raw, un-engineered features for the baseline
features = ["amt", "city_pop"]
target = "is_fraud"

X_train = sparkov_train[features]
y_train = sparkov_train[target]
X_test = sparkov_test[features]
y_test = sparkov_test[target]

# 4. Scale the features - Logistic Regression works better on similar-sized numbers
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn the scaling from train only
X_test_scaled = scaler.transform(X_test)         # apply that same scaling to test

# 5. Train a plain Logistic Regression - default settings, no class weighting
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 6. Score it with the project's three metrics (not accuracy - see EDA notes on why)
from sklearn.metrics import average_precision_score, fbeta_score, matthews_corrcoef

y_prob = model.predict_proba(X_test_scaled)[:, 1]   # predicted fraud probability
y_pred = (y_prob >= 0.5).astype(int)                # turn into 0/1 at the default threshold

print("PR-AUC:", average_precision_score(y_test, y_prob))
print("F2:", fbeta_score(y_test, y_pred, beta=2))
print("MCC:", matthews_corrcoef(y_test, y_pred))

Train fraud rate: 0.005788651743883394
Test fraud rate: 0.0038598644278853163
PR-AUC: 0.13658130213813538
F2: 0.0
MCC: -0.0015826530982826614


We trained a plain Logistic Regression using only two raw columns — transaction amount and city population — with no feature engineering, no class rebalancing, and the default 0.5 decision threshold. This gives us a naive baseline: PR-AUC 0.137, F2 0.0, MCC -0.0016. The test set's fraud rate (0.386%) came in lower than train's (0.579%), a first sign of how skewed this problem is before any modelling even starts. A PR-AUC of 0.137 is far below ULB's 0.73, but that's expected — ULB's raw features already carry more embedded signal, while here we deliberately held back the strongest raw signal (transaction category) so it can be introduced later as an engineered feature. Even so, 0.137 is meaningfully above the 0.386% base rate, so the two weak features aren't contributing nothing. F2 and MCC both landing near zero show that, at the default threshold, the model effectively fails to catch any fraud. We treat this row as the fixed comparison point that every subsequent feature group gets measured against.

In [7]:
print("Predicted positives:", y_pred.sum(), "out of", len(y_pred))

Predicted positives: 359 out of 555719


To understand exactly why F2 came out at 0.0, we counted how many transactions the model flagged as fraud: 359 out of 555,719. The test set contains roughly 2,145 actual fraud cases (0.386% of 555,719), so an F2 of 0.0 tells us that none of the 359 flagged transactions were real frauds — the model missed every genuine fraud case and raised 359 false alarms instead. This is expected for a model built on two features with no direct link to fraud behavior, scored at an untuned threshold, and it previews the two directions the rest of this analysis addresses: richer features, and imbalance-aware training so real fraud cases actually surface.

In [8]:
# add category risk as a feature, then re-run the same baseline steps

from src.utils import fit_category_risk, apply_category_risk

# 1. Learn the fraud rate per category from TRAIN ONLY
category_risk_lookup = fit_category_risk(sparkov_train, category_column="category", label_column="is_fraud")

# 2. Apply that same lookup to both train and test (test never influences the numbers)
sparkov_train["category_risk"] = apply_category_risk(sparkov_train, "category", category_risk_lookup)
sparkov_test["category_risk"] = apply_category_risk(sparkov_test, "category", category_risk_lookup)

# 3. Same six steps as Cell 2, just with category_risk added to the feature list
features = ["amt", "city_pop", "category_risk"]
target = "is_fraud"

X_train = sparkov_train[features]
y_train = sparkov_train[target]
X_test = sparkov_test[features]
y_test = sparkov_test[target]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

from sklearn.metrics import average_precision_score, fbeta_score, matthews_corrcoef
y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("PR-AUC:", average_precision_score(y_test, y_prob))
print("F2:", fbeta_score(y_test, y_pred, beta=2))
print("MCC:", matthews_corrcoef(y_test, y_pred))

PR-AUC: 0.1393967891206238
F2: 0.0
MCC: -0.0014871395291730385


Adding category risk moved PR-AUC from 0.1366 to 0.1394 — a small increase, and F2 and MCC are effectively unchanged (F2 still 0.0, MCC still close to zero). That's a smaller lift than the EDA finding would suggest — category showed an 11x spread in fraud rate between the highest and lowest categories, which reads as a strong signal on its own. So before treating this as "category risk barely helps" and moving on, it's worth checking whether that's the true picture or whether something in how the feature is being used is muting it.

There's a plausible non-bug explanation: even the riskiest category tops out at about 1.76% fraud rate, against a 0.386% test base rate. That's a real signal, but it's still a low absolute number — a feature that only ever nudges predicted probability up to ~1.76% doesn't give a linear model much room to achieve high precision at reasonable recall, which is specifically what PR-AUC measures. An 11x relative spread and a strong PR-AUC contribution aren't automatically the same thing.

In [9]:
# isolate category_risk to see how much signal it carries by itself
features_solo = ["category_risk"]

X_train_solo = sparkov_train[features_solo]
X_test_solo = sparkov_test[features_solo]

scaler_solo = StandardScaler()
X_train_solo_scaled = scaler_solo.fit_transform(X_train_solo)
X_test_solo_scaled = scaler_solo.transform(X_test_solo)

model_solo = LogisticRegression(max_iter=1000)
model_solo.fit(X_train_solo_scaled, y_train)

y_prob_solo = model_solo.predict_proba(X_test_solo_scaled)[:, 1]
y_pred_solo = (y_prob_solo >= 0.5).astype(int)

print("category_risk alone - PR-AUC:", average_precision_score(y_test, y_prob_solo))
print("category_risk alone - F2:", fbeta_score(y_test, y_pred_solo, beta=2))
print("category_risk alone - MCC:", matthews_corrcoef(y_test, y_pred_solo))

# and the actual per-category values, to see the real spread this feature carries
print(sparkov_train.groupby("category")["category_risk"].first().sort_values(ascending=False))

category_risk alone - PR-AUC: 0.009177127556498067
category_risk alone - F2: 0.0
category_risk alone - MCC: 0.0
category
shopping_net      0.017561
misc_net          0.014458
grocery_pos       0.014098
shopping_pos      0.007225
gas_transport     0.004694
misc_pos          0.003139
grocery_net       0.002948
travel            0.002864
entertainment     0.002478
personal_care     0.002424
kids_pets         0.002114
food_dining       0.001651
home              0.001608
health_fitness    0.001549
Name: category_risk, dtype: float64


On its own, category_risk scores PR-AUC 0.0092 — only about 2.4x the test base rate (0.386%), and far below the 0.1366 that amt+city_pop alone achieved. So this isn't a bug or something suppressing the feature when combined — category really is a much weaker ranking signal on its own than the earlier descriptive stat suggested. The category values themselves look correct and match the EDA's spread (1.76% for shopping_net down to 0.15% for health_fitness), so the data is right; it's the gap between "statistically significant spread in aggregate fraud rate" and "strong individual predictive power" that explains the small lift. An 11x relative difference between categories still means every category tops out under 2% fraud rate — not enough on its own to let the model reach high precision at meaningful recall, which is what PR-AUC measures. Combined with amt/city_pop, category_risk still adds a small amount (0.1366 to 0.1394), just less than the raw spread implied.

One more thing worth flagging now, since it'll apply to every remaining cell in this stage: F2 and MCC came back at exactly 0.0 this time (not slightly negative like before) — that means the model predicted zero positive cases at all with category_risk as the only feature. At this severe an imbalance, with no class weighting or threshold tuning in play, a plain 0.5 threshold will keep producing 0 or near-0 predicted positives almost regardless of which features we add — so F2 and MCC are likely to stay pinned near zero through the rest of this feature-engineering stage. PR-AUC is the metric actually doing the work of showing whether each feature group helps; F2/MCC only really become informative once imbalance handling (resampling, class weights, threshold tuning) enters the picture later. Worth stating plainly in the write-up so a reader doesn't mistake a flat F2/MCC for "the feature didn't help."

In [10]:
# hour-of-day - test raw hour vs an is_night flag, same six-step shape as before

# 1. Extract the raw hour from the timestamp
sparkov_train["hour"] = sparkov_train["trans_date_trans_time"].dt.hour
sparkov_test["hour"] = sparkov_test["trans_date_trans_time"].dt.hour

# 2. Build an is_night flag - EDA found the fraud-rate spike is specifically 10pm-3am
night_hours = [22, 23, 0, 1, 2, 3]
sparkov_train["is_night"] = sparkov_train["hour"].isin(night_hours).astype(int)
sparkov_test["is_night"] = sparkov_test["hour"].isin(night_hours).astype(int)

target = "is_fraud"
y_train = sparkov_train[target]
y_test = sparkov_test[target]

# 3. Test A - raw hour (0-23) as a plain number
features_raw_hour = ["amt", "city_pop", "category_risk", "hour"]
X_train = sparkov_train[features_raw_hour]
X_test = sparkov_test[features_raw_hour]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("Raw hour   - PR-AUC:", average_precision_score(y_test, y_prob))
print("Raw hour   - F2:", fbeta_score(y_test, y_pred, beta=2))
print("Raw hour   - MCC:", matthews_corrcoef(y_test, y_pred))

# 4. Test B - is_night flag instead of raw hour
features_is_night = ["amt", "city_pop", "category_risk", "is_night"]
X_train = sparkov_train[features_is_night]
X_test = sparkov_test[features_is_night]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("is_night   - PR-AUC:", average_precision_score(y_test, y_prob))
print("is_night   - F2:", fbeta_score(y_test, y_pred, beta=2))
print("is_night   - MCC:", matthews_corrcoef(y_test, y_pred))

Raw hour   - PR-AUC: 0.12995423457338073
Raw hour   - F2: 0.0
Raw hour   - MCC: -0.0014587044725662018
is_night   - PR-AUC: 0.22296414849693877
is_night   - F2: 0.0
is_night   - MCC: -0.001402597165957869


This is a clean, unambiguous result. Raw hour actually made things slightly worse — PR-AUC dropped from 0.1394 (with category_risk only) to 0.1300 with raw hour added. Forcing a linear model to treat hour 23 and hour 1 as numerically distant, when they're both part of the same overnight risk window, doesn't just fail to help — it actively misleads the model, since it spends model capacity trying to fit a straight-line trend to a pattern that isn't one. The is_night flag, by contrast, jumped PR-AUC to 0.2230 — up from 0.1394, a genuinely large gain (roughly +60% relative), because it hands the model exactly the shape the EDA found: overnight is risky, daytime isn't, with no in-between gradient to misrepresent. F2 and MCC stay near zero in both cases, consistent with what we already established — those two won't move until imbalance handling changes the threshold behavior.

This is worth keeping as its own point in the write-up beyond just "hour-of-day helps": it's a direct, evidenced demonstration that how a feature is encoded can matter as much as whether the underlying signal exists — the same information, encoded two different ways, went from mildly harmful to the single biggest lift so far.

Going forward, is_night replaces raw hour as the canonical feature — raw hour is dropped from the running feature set since we now have direct evidence it hurts rather than helps.

In [11]:
# look at the amount distribution before deciding how to engineer it
sparkov_train.groupby("is_fraud")["amt"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,67.667110,154.007971,1.00,9.6100,47.280,82.540,28948.90
1,7506.0,531.320092,390.560070,1.06,245.6625,396.505,900.875,1376.04


Two numbers here matter most. Legit transactions' 75th percentile is $82.54, while fraud transactions' 25th percentile is $245.66 — meaning three-quarters of legit transactions sit below the value that even the cheapest quarter of fraud transactions start at. That's a strong, clean separation in the middle of the distribution. But the story isn't a simple "bigger = riskier" one: legit transactions have a long tail up to $28,948.90 (a handful of large but legitimate purchases), while fraud is capped at $1,376.04 — fraud in this data never goes above that. So the real pattern the EDA called "two distinct bands" is really a range: fraud amounts cluster roughly between $100 and $1,400, while legit amounts are either much smaller or, less often, much larger than that range.


In [13]:
# test amount encodings against the is_night-based feature set

import numpy as np

target = "is_fraud"
y_train = sparkov_train[target]
y_test = sparkov_test[target]

# candidate 1: log-transform, tames the long legit tail
sparkov_train["amt_log"] = np.log1p(sparkov_train["amt"])
sparkov_test["amt_log"] = np.log1p(sparkov_test["amt"])

# candidate 2: flag for the $100-$1400 range fraud actually clusters in
sparkov_train["amt_in_fraud_range"] = sparkov_train["amt"].between(100, 1400).astype(int)
sparkov_test["amt_in_fraud_range"] = sparkov_test["amt"].between(100, 1400).astype(int)

# same six steps as every cell so far - scale, fit, score - just reusable
# since we're comparing three feature lists in this one cell
def scale_fit_score(features):
    X_train = sparkov_train[features]
    X_test = sparkov_test[features]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "PR-AUC": average_precision_score(y_test, y_prob),
        "F2": fbeta_score(y_test, y_pred, beta=2),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }

print("log1p(amt)        ->", scale_fit_score(["amt_log", "city_pop", "category_risk", "is_night"]))
print("amt_in_fraud_range ->", scale_fit_score(["amt_in_fraud_range", "city_pop", "category_risk", "is_night"]))

log1p(amt)        -> {'PR-AUC': np.float64(0.3060769389608977), 'F2': 0.12955960854092527, 'MCC': np.float64(0.24672900068727713)}
amt_in_fraud_range -> {'PR-AUC': np.float64(0.0990439204621514), 'F2': 0.0, 'MCC': 0.0}


log1p(amt) is a clear, large win: PR-AUC jumped from 0.2230 to 0.3061, and — genuinely notable — F2 and MCC finally moved off zero, to 0.1296 and 0.2467. Every feature group so far only shifted the model's ranking of transactions (PR-AUC), while F2/MCC stayed pinned near zero because the model's predicted probabilities never crossed the 0.5 threshold. Log-transformed amount is the first feature strong enough to actually flip some real predictions past that threshold — meaning at the plain default cutoff, with no resampling or class weighting yet, the model is now catching some real fraud cases outright. Worth stating directly in the write-up: this is the point where amount, once properly scaled, stops being a marginal contributor and becomes a primary driver.

The banded flag went the opposite direction — PR-AUC dropped to 0.0990, worse than even plain raw amount, and F2/MCC collapsed back to exactly zero. The reason ties back to what the describe() output actually showed: amount's relationship to fraud isn't a step function the way hour's was. Day-vs-night was genuinely binary (that's why a flag worked so well for it), but amount's relationship is graded — a $1,200 transaction is more suspicious than a $150 one, both potentially inside the same $100-$1,400 band. Squashing that into a single bit throws away exactly the information that matters, and worse, it lumps in a chunk of ordinary legit transactions that also happen to fall in that dollar range, adding noise instead of signal. Good, concrete example for the write-up of why the shape of a feature's true relationship to the target — step-like versus continuous — should drive the encoding choice, not just "does a difference exist."

Running feature set going forward: amt_log, city_pop, category_risk, is_night → PR-AUC 0.3061, F2 0.1296, MCC 0.2467. The banded flag is dropped.

In [15]:
# implied velocity - compute across the full timeline so test's first
# transaction per cardholder still has a real "previous transaction" from train
import pandas as pd

from src.utils import add_prev_txn_features

# 1. Tag rows so we can split back apart afterward, then stick train+test together
sparkov_train["_split"] = "train"
sparkov_test["_split"] = "test"
combined = pd.concat([sparkov_train, sparkov_test], ignore_index=True)

# 2. Compute velocity across the full timeline - only looks backward per cardholder,
#    so this doesn't leak any test-set fraud label into anything
combined = add_prev_txn_features(
    combined,
    id_column="cc_num",
    time_column="trans_date_trans_time",
    lat_column="merch_lat",
    lon_column="merch_long",
)

# 3. Split back into train/test using the tag from step 1
sparkov_train = combined[combined["_split"] == "train"].drop(columns=["_split"])
sparkov_test = combined[combined["_split"] == "test"].drop(columns=["_split"])

# 4. Look at the result before deciding how to scale it
print(sparkov_train[["dist_from_prev_txn_km", "time_gap_hours", "implied_velocity_kmh"]].describe())
print("Missing in train:", sparkov_train["implied_velocity_kmh"].isna().sum())
print("Missing in test:", sparkov_test["implied_velocity_kmh"].isna().sum())

       dist_from_prev_txn_km  time_gap_hours  implied_velocity_kmh
count           1.295692e+06    1.295692e+06          1.295692e+06
mean            1.035944e+02    9.023615e+00          1.741713e+02
std             5.039868e+01    1.316694e+01          2.757404e+03
min             8.069198e-02    0.000000e+00          3.682374e-03
25%             6.438102e+01    1.670278e+00          7.599740e+00
50%             1.007818e+02    4.600000e+00          2.001599e+01
75%             1.397061e+02    1.117306e+01          5.876024e+01
max             2.859922e+02    3.726308e+02          7.362208e+05
Missing in train: 983
Missing in test: 16


The missing counts confirm the fix worked as intended. 983 missing in train matches exactly the EDA's count of unique cardholders in the training set — these are each cardholder's genuine first-ever transaction, where no "previous transaction" can exist by definition, so NaN there is correct, not a bug. The 16 missing in test are almost certainly new cardholders who don't appear in train at all — their first-ever transaction happens to land in the test period, so there's still no real previous transaction to reference even with the combined-timeline fix. Both are the honest, unavoidable version of "no prior history exists," not the boundary artifact we fixed — that fix is confirmed by the fact that test's missing count is now 16, not the ~983 it would have been if computed on the split test set alone with no continuity from train.

The distribution is exactly what the EDA warned about. dist_from_prev_txn_km and time_gap_hours both look like sane, real-world numbers (max 286 km, max ~15.5 days between transactions). But implied_velocity_kmh has a mean of 174 km/h against a median of just 20 km/h, with a max of 736,220 km/h — a physically impossible number, confirming this is the merchant-registration-location artifact flagged in the EDA rather than real travel. Feeding that raw into a StandardScaler would let a handful of these extreme rows dominate the scaling and drown out the genuine signal — the same class of problem we solved for amt with log1p, just far more extreme here (8 orders of magnitude between min and max, versus roughly 4 for amount).

In [16]:
# implied velocity - fill debut-transaction NaNs, then test two ways
# of taming the extreme outliers (up to 736,000+ km/h, a known merchant-location artifact)

target = "is_fraud"
y_train = sparkov_train[target]
y_test = sparkov_test[target]

# 1. No previous transaction exists yet for these rows - fill with 0 ("no prior info")
for col in ["dist_from_prev_txn_km", "time_gap_hours", "implied_velocity_kmh"]:
    sparkov_train[col] = sparkov_train[col].fillna(0)
    sparkov_test[col] = sparkov_test[col].fillna(0)

# 2. Candidate A: log-transform, same fix that worked for amount
sparkov_train["velocity_log"] = np.log1p(sparkov_train["implied_velocity_kmh"])
sparkov_test["velocity_log"] = np.log1p(sparkov_test["implied_velocity_kmh"])

# 3. Candidate B: clip at the 99th percentile, learned from train only
cap = sparkov_train["implied_velocity_kmh"].quantile(0.99)
sparkov_train["velocity_clipped"] = sparkov_train["implied_velocity_kmh"].clip(upper=cap)
sparkov_test["velocity_clipped"] = sparkov_test["implied_velocity_kmh"].clip(upper=cap)
print("99th percentile cap (from train):", cap)

def scale_fit_score(features):
    X_train = sparkov_train[features]
    X_test = sparkov_test[features]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "PR-AUC": average_precision_score(y_test, y_prob),
        "F2": fbeta_score(y_test, y_pred, beta=2),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }

print("velocity_log     ->", scale_fit_score(["amt_log", "city_pop", "category_risk", "is_night", "velocity_log"]))
print("velocity_clipped ->", scale_fit_score(["amt_log", "city_pop", "category_risk", "is_night", "velocity_clipped"]))

99th percentile cap (from train): 1955.7611217790948
velocity_log     -> {'PR-AUC': np.float64(0.34571374713814906), 'F2': 0.20970214629872974, 'MCC': np.float64(0.35087176785522034)}
velocity_clipped -> {'PR-AUC': np.float64(0.3311856602611441), 'F2': 0.14471057884231536, 'MCC': np.float64(0.268150443094085)}


velocity_log wins clearly across all three metrics — PR-AUC 0.3457 vs 0.3312, F2 0.2097 vs 0.1447, MCC 0.3509 vs 0.2682 — and both beat the 0.3061 reference, so velocity is adding real value either way. Same underlying reason as amount: velocity's relationship to fraud is graded and continuous (the EDA explicitly found it "holds across percentiles, not just driven by outliers"), so log-transforming preserves that graded structure across the whole range while still taming the extreme merchant-location artifacts. Clipping only handles the very top of the tail — it does nothing for the more moderate compression needed across the rest of the distribution, so it recovers less of the signal. velocity_log becomes the final feature.

That completes all four feature groups. Final running result: amt_log, city_pop, category_risk, is_night, velocity_log → PR-AUC 0.3457, F2 0.2097, MCC 0.3509, up from a naive baseline of 0.1366 / 0.0 / -0.0016.

In [17]:
# the full uplift table, built cleanly in one place

FEATURE_STEPS = {
    "baseline (raw amt, city_pop)": ["amt", "city_pop"],
    "+ category_risk": ["amt", "city_pop", "category_risk"],
    "+ is_night": ["amt", "city_pop", "category_risk", "is_night"],
    "+ amt_log (replaces raw amt)": ["amt_log", "city_pop", "category_risk", "is_night"],
    "+ velocity_log": ["amt_log", "city_pop", "category_risk", "is_night", "velocity_log"],
}

rows = {}
for name, features in FEATURE_STEPS.items():
    rows[name] = scale_fit_score(features)

uplift_table = pd.DataFrame(rows).T

baseline_scores = uplift_table.iloc[0]
for metric in ["PR-AUC", "F2", "MCC"]:
    uplift_table[f"{metric} delta"] = uplift_table[metric] - baseline_scores[metric]

uplift_table

,PR-AUC,F2,MCC,PR-AUC delta,F2 delta,MCC delta
"baseline (raw amt, city_pop)",0.136581,0.000000,-0.001583,0.000000,0.000000,0.000000
+ category_risk,0.139397,0.000000,-0.001487,0.002815,0.000000,0.000096
+ is_night,0.222964,0.000000,-0.001403,0.086383,0.000000,0.000180
+ amt_log (replaces raw amt),0.306077,0.129560,0.246729,0.169496,0.129560,0.248312
+ velocity_log,0.345714,0.209702,0.350872,0.209132,0.209702,0.352454


In [18]:
# drop the columns we tested and didn't keep
sparkov_train = sparkov_train.drop(columns=["hour", "amt_in_fraud_range", "velocity_clipped"], errors="ignore")
sparkov_test = sparkov_test.drop(columns=["hour", "amt_in_fraud_range", "velocity_clipped"], errors="ignore")

print(sparkov_train.columns.tolist())

['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'category_risk', 'is_night', 'amt_log', 'dist_from_prev_txn_km', 'time_gap_hours', 'implied_velocity_kmh', 'velocity_log']
